# SmartMoney · Fase 5 — Simulación del bandido Thompson Sampling

**Trabajo de grado · Universidad UNAB · Componente de IA del motor de recomendaciones**

---

## Qué hace este cuaderno

El motor de recomendaciones de SmartMoney detecta hallazgos financieros de seis
tipos, los puntúa y muestra al usuario los mejor puntuados. El puntaje de cada
hallazgo es:

```
puntaje = peso_tipo × severidad × impacto × confianza
```

Antes, `peso_tipo` era una constante fija por tipo. En la Fase 5 se reemplazó
por un **bandido multi-brazo con muestreo de Thompson y prior jerárquico**: el
peso de cada tipo se aprende de la retroalimentación real (botones «Útil» /
«No útil») y se comparte información entre usuarios.

Este cuaderno **simula** ese motor para responder preguntas que el piloto real,
con pocos usuarios, no puede responder con potencia estadística:

1. ¿Cuánto mejora el bandido jerárquico frente al peso fijo y frente a un
   bandido ingenuo (sin prior)?
2. ¿En cuántas interacciones aprende, y cuánto le falta para el techo teórico
   (oráculo)?
3. ¿Qué potencia tendría el A/B real del piloto (20 usuarios, 10 interacciones)?
4. ¿Qué tan sensible es al parámetro `M` (fuerza del prior del usuario)?
5. ¿Cuánto depende el resultado de cuánto se parezcan los usuarios entre sí?

## Políticas comparadas

| Política | Descripción |
|---|---|
| `control` | `peso_tipo = μ_diseño` fijo (lo que había antes del bandido). |
| `aleatoria` | Elige un hallazgo disponible al azar. |
| `thompson_sin_prior` | Beta(1 + útiles, 1 + no útiles): solo el historial del usuario. |
| `thompson_jerarquico` | **La política de SmartMoney**: prior poblacional (leave-one-out) + historial del usuario. Se evalúa con `M = 1, 2, 4 y 10`. |
| `oraculo` | Conoce la probabilidad real y elige la mejor opción. Es un **techo**, no una política implementable. |

## Dos escenarios de población

El rendimiento del prior jerárquico depende de cuánto se parezcan los usuarios
entre sí, porque el prior *es* la media de la población. Por eso el cuaderno
estudia **dos poblaciones** con los mismos usuarios, el mismo entorno y las
mismas semillas:

| Escenario | Nombre | Idea |
|---|---|---|
| **A** | «perfiles opuestos» | Cada perfil valora tipos distintos y opuestos: casi no hay terreno común. |
| **B** | «población con tendencia común» | Los tres perfiles comparten una base (todos valoran más `sobregasto` y `meta`) y solo cambian en el matiz. |

En A la media de la población es un punto medio que **no describe a nadie**; en
B describe razonablemente a la mayoría. Esa diferencia es la que se quiere
medir.

## Cómo leer el resto

* Las celdas de código son autocontenidas: solo numpy, pandas y matplotlib.
* Todo es **reproducible**: se fijan semillas con `numpy.random.default_rng`.
* Las figuras se guardan como PNG (dpi 200) en `ia/fase5/` y los resultados en
  tres CSV, para citarlos en el documento de tesis.


## 1. El bandido de SmartMoney: fórmulas exactas

Las expresiones siguientes se copian literalmente de
`supabase/functions/motor-recomendaciones/bandido.ts` y `config.ts`.
La simulación las replica **sin cambios**.

### Media poblacional del tipo (prior jerárquico)

Para cada tipo `k`:

```
μ_poblacion(k) = (utiles_k + C · μ_diseno(k)) / (respuestas_k + C)
```

* `utiles_k` y `respuestas_k` son los conteos **de la población**, calculados
  **sin** las respuestas del propio usuario (leave-one-out):
  `utiles_poblacion = utiles_globales − utiles_usuario` (nunca negativo).
* `C = C_DISENO = 4` es la fuerza del prior de diseño.
* `μ_diseno` es un supuesto de diseño:
  `{sobregasto 0.80, meta 0.75, anomalía 0.65, logro 0.55, lección 0.50}`.
* Si nadie ha respondido, `μ_poblacion(k) = μ_diseno(k)` exactamente: ese es el
  punto de partida del sistema.

### Parámetros Beta del usuario

```
α(k) = M · μ_poblacion(k) + utiles_usuario(k)
β(k) = M · (1 − μ_poblacion(k)) + no_utiles_usuario(k)
```

con `M = M_PRIOR = 4`. Sin clics del usuario `α + β = M` y la media `α/(α+β)`
coincide con la media poblacional. Cada clic «arrastra» la media hacia la tasa
observada del usuario.

### Peso del tipo

* Grupo **bandido**: `peso_tipo(k) ≈ Beta(α(k), β(k))` (una muestra por
  interacción; así se explora de forma natural sin dejar de explotar).
* Grupo **control**: `peso_tipo(k) = PESO_TIPO_CONTROL(k) = μ_diseno(k)`.

### Selección

`puntaje(k) = peso_tipo(k) × severidad(k) × impacto(k) × confianza(k)` y se
muestra el tipo disponible con mayor puntaje. El tipo `riesgo_fin_de_mes` queda
fuera del bandido y por eso no participa en esta simulación.


## 2. Supuestos de diseño: probabilidad real de «útil» por perfil y por escenario

Como todavía no existe un dataset real de respuestas, la simulación define
**tres perfiles de usuario** con una probabilidad real de responder «útil» a
cada tipo de recomendación. Son **supuestos de diseño** explícitos: definen los
escenarios que se estudian, y por eso se documentan y se pueden cambiar.

### Escenario A · «perfiles opuestos»

| Tipo | alertas | metas | aprendiz |
|---|---|---|---|
| sobregasto | 0.80 | 0.40 | 0.45 |
| anomalía | 0.75 | 0.35 | 0.30 |
| meta | 0.40 | 0.85 | 0.35 |
| lección | 0.20 | 0.30 | 0.80 |
| logro | 0.30 | 0.60 | 0.70 |

Cada perfil valora un grupo de tipos distinto y **desprecia justo lo que valora
otro** (el «aprendiz» rechaza las alertas; el de «alertas» rechaza las
lecciones). No existe un usuario «promedio»: el prior poblacional es una mezcla
de extremos que no describe a nadie.

* **alertas**: le importa saber dónde se está pasando de la raya
  (sobregasto y anomalías altos; lecciones y logros bajos).
* **metas**: le importa ahorrar y cumplir objetivos (meta y logro altos).
* **aprendiz**: quiere aprender a manejar mejor su dinero (lección y logro
  altos).

### Escenario B · «población con tendencia común»

| Tipo | alertas | metas | aprendiz |
|---|---|---|---|
| sobregasto | 0.85 | 0.65 | 0.60 |
| anomalía | 0.65 | 0.40 | 0.30 |
| meta | 0.70 | 0.85 | 0.65 |
| lección | 0.25 | 0.30 | 0.70 |
| logro | 0.35 | 0.55 | 0.50 |

Aquí los tres perfiles **comparten una base**: todos valoran por encima del peso
de diseño tanto el sobregasto como la meta, y todos puntúan bajo la lección
salvo el perfil «aprendiz». La variación personal existe, pero es un matiz sobre
una tendencia común. Es el caso que **favorece al prior jerárquico**: la media
de la población sí describe razonablemente a un usuario nuevo.

### Por qué se comparan los dos

El prior jerárquico «pide prestada» información de la población. Ese préstamo
sale caro si la población es heterogénea (Escenario A: la media no representa a
nadie) y sale bien si la población tiene una tendencia común (Escenario B). La
diferencia entre escenarios cuantifica cuánto depende la política de ese
supuesto.

Cada perfil tiene 100 usuarios (300 en total) y **30 interacciones** (≈ un mes,
una recomendación por día). Las probabilidades de cada escenario se declaran en
la celda siguiente.


In [ ]:
# ============================================================================
# SmartMoney · Fase 5 · Simulación del bandido Thompson Sampling
# ----------------------------------------------------------------------------
# Cuaderno autocontenido: solo numpy, pandas y matplotlib. Se puede ejecutar en
# Google Colab sin subir ningún archivo externo.
# Todas las constantes y fórmulas replican EXACTAMENTE
# supabase/functions/motor-recomendaciones/config.ts y bandido.ts.
# ============================================================================

import math
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Reproducibilidad --------------------------------------------------------
SEMILLA_BASE = 20261022
N_SEMILLAS = 50
SEMILLAS = [SEMILLA_BASE + i for i in range(N_SEMILLAS)]
Z_IC = 1.96  # nivel de confianza del 95 %

# --- Constantes replicadas del motor (config.ts) -----------------------------
TIPOS = [
    "sobregasto_categoria",
    "gasto_anomalo",
    "aporte_meta",
    "leccion_sugerida",
    "logro",
]
ETIQUETA = {
    "sobregasto_categoria": "sobregasto",
    "gasto_anomalo": "anomalía",
    "aporte_meta": "meta",
    "leccion_sugerida": "lección",
    "logro": "logro",
}

# MU_DISENO (config.ts): supuesto de diseño, no una tasa observada.
MU_DISENO = {
    "sobregasto_categoria": 0.80,
    "aporte_meta": 0.75,
    "gasto_anomalo": 0.65,
    "logro": 0.55,
    "leccion_sugerida": 0.50,
}
MU_VEC = np.array([MU_DISENO[t] for t in TIPOS])   # orden = TIPOS
PESO_CONTROL = MU_VEC.copy()                       # PESO_TIPO_CONTROL = {...MU_DISENO}
C_DISENO = 4   # fuerza del prior poblacional (respuestas virtuales del diseño)
M_PRIOR = 4    # fuerza del prior del usuario (respuestas virtuales de la población)
M_VALORES = [1, 2, 4, 10]   # fuerzas del prior evaluadas en la sensibilidad (sección 12)

# --- Parámetros del experimento ---------------------------------------------
PERFILES = ["alertas", "metas", "aprendiz"]
USUARIOS_POR_PERFIL = 100
N_USUARIOS = USUARIOS_POR_PERFIL * len(PERFILES)   # 300 usuarios
T_INTER = 30                                       # ≈ un mes: 1 recomendación/día
K = len(TIPOS)
P_DISPONIBLE = 0.5            # prob. de que un tipo aparezca como hallazgo disponible
RANGO_FEATURES = (0.3, 1.0)   # severidad, impacto y confianza ~ U(0.3, 1.0)

# --- Estilo de las figuras ---------------------------------------------------
ORDEN = ["control", "aleatoria", "thompson_sin_prior", "thompson_jerarquico", "oraculo"]
NOMBRES = {
    "control": "Control (peso fijo μ_diseño)",
    "aleatoria": "Aleatoria",
    "thompson_sin_prior": "Thompson sin prior",
    "thompson_jerarquico": "Thompson jerárquico (SmartMoney)",
    "oraculo": "Oráculo (techo)",
}
COLORES = {
    "control": "#6b7280",
    "aleatoria": "#9ca3af",
    "thompson_sin_prior": "#f59e0b",
    "thompson_jerarquico": "#2563eb",
    "oraculo": "#059669",
}
ESTILOS = {
    "control": "--",
    "aleatoria": ":",
    "thompson_sin_prior": "-.",
    "thompson_jerarquico": "-",
    "oraculo": "--",
}

# --- Carpeta de salida -------------------------------------------------------
DIR_SALIDA = "ia/fase5"
os.makedirs(DIR_SALIDA, exist_ok=True)


def ruta(nombre):
    return os.path.join(DIR_SALIDA, nombre)


print("numpy", np.__version__, "| pandas", pd.__version__)
print("Usuarios:", N_USUARIOS, "| Interacciones:", T_INTER, "| Semillas:", N_SEMILLAS)
print("Tipos del bandido:", TIPOS)
print("Carpeta de salida:", DIR_SALIDA)


In [ ]:
# Probabilidad REAL de "útil" de cada usuario, por perfil y por tipo.
# Son supuestos de diseño (no provienen de datos observados): definen el escenario.
#
# ESCENARIO A · "perfiles opuestos": cada perfil valora tipos distintos y opuestos.
R_PROB_A = {
    "alertas": {
        "sobregasto_categoria": 0.80, "gasto_anomalo": 0.75,
        "aporte_meta": 0.40, "logro": 0.30, "leccion_sugerida": 0.20,
    },
    "metas": {
        "aporte_meta": 0.85, "logro": 0.60, "sobregasto_categoria": 0.40,
        "gasto_anomalo": 0.35, "leccion_sugerida": 0.30,
    },
    "aprendiz": {
        "leccion_sugerida": 0.80, "logro": 0.70, "sobregasto_categoria": 0.45,
        "aporte_meta": 0.35, "gasto_anomalo": 0.30,
    },
}

# ESCENARIO B · "población con tendencia común": los tres perfiles comparten una
# base (todos valoran más sobregasto y meta) y solo cambian en el matiz.
R_PROB_B = {
    "alertas": {
        "sobregasto_categoria": 0.85, "gasto_anomalo": 0.65,
        "aporte_meta": 0.70, "logro": 0.35, "leccion_sugerida": 0.25,
    },
    "metas": {
        "sobregasto_categoria": 0.65, "gasto_anomalo": 0.40,
        "aporte_meta": 0.85, "logro": 0.55, "leccion_sugerida": 0.30,
    },
    "aprendiz": {
        "sobregasto_categoria": 0.60, "gasto_anomalo": 0.30,
        "aporte_meta": 0.65, "logro": 0.50, "leccion_sugerida": 0.70,
    },
}

R_PROB_ESCENARIO = {"A": R_PROB_A, "B": R_PROB_B}
ESCENARIOS = ["A", "B"]
ESC_A, ESC_B = "A", "B"   # atajos
NOMBRE_ESCENARIO = {
    "A": 'Escenario A · «perfiles opuestos»',
    "B": 'Escenario B · «población con tendencia común»',
}

# Perfil de cada uno de los 300 usuarios (100 por perfil, en orden).
PERFIL_USUARIO = np.repeat(np.arange(len(PERFILES)), USUARIOS_POR_PERFIL)


def matriz_p_real(escenario):
    """Matriz (N_USUARIOS x K) con la probabilidad real de "útil" de cada usuario."""
    R = R_PROB_ESCENARIO[escenario]
    return np.array([[R[PERFILES[g]][t] for t in TIPOS] for g in PERFIL_USUARIO])


# P_REAL[escenario] -> matriz (N_USUARIOS x K)
P_REAL = {esc: matriz_p_real(esc) for esc in ESCENARIOS}

for esc in ESCENARIOS:
    tabla_perfiles = pd.DataFrame(
        {PERFILES[g]: [R_PROB_ESCENARIO[esc][PERFILES[g]][t] for t in TIPOS]
         for g in range(len(PERFILES))},
        index=[ETIQUETA[t] for t in TIPOS],
    )
    print(NOMBRE_ESCENARIO[esc])
    print(tabla_perfiles.round(2))
    print()

print("Usuarios por perfil:", dict(zip(PERFILES, np.bincount(PERFIL_USUARIO))))


## 3. El entorno simulado

Para que la comparación sea justa, **las cinco políticas ven exactamente los
mismos hallazgos**: el entorno (qué tipos están disponibles y con qué
severidad, impacto y confianza) se genera **una sola vez por semilla** y se
reutiliza en todas las políticas.

En cada interacción de cada usuario:

1. **Disponibilidad**: cada tipo aparece como hallazgo disponible con
   probabilidad 0.5. Si por azar no hay ninguno, se habilita uno al azar: en el
   motor real nunca se ofrece una lista vacía.
2. **Atributos**: `severidad`, `impacto` y `confianza` ~ Uniforme(0.3, 1.0).
3. **Elección**: la política calcula el puntaje de cada tipo disponible y elige
   el de mayor puntaje.
4. **Respuesta**: el usuario responde «útil» con su probabilidad real para el
   tipo elegido. Para reducir varianza, esa decisión usa un **número aleatorio
   común** a todas las políticas (common random numbers): así las diferencias
   entre políticas se deben a *qué* recomiendan y no a ruido adicional.

Esta es la traducción directa de `puntaje.ts` + `bandido.ts` al lenguaje de la
simulación.


In [ ]:
def generar_ambiente(rng, n_usuarios, t_inter, k):
    """Genera el entorno de una semilla: el MISMO para todas las políticas.

    Devuelve:
      disponible : (n, t, k) bool   -> el tipo aparece como hallazgo disponible
      severidad  : (n, t, k) float
      impacto    : (n, t, k) float
      confianza  : (n, t, k) float
      u_respuesta: (n, t)   float   -> número aleatorio común para decidir "útil"
    """
    disponible = rng.random((n_usuarios, t_inter, k)) < P_DISPONIBLE

    # Garantiza que nunca haya una lista vacía de hallazgos.
    vacios = ~disponible.any(axis=2)
    if vacios.any():
        filas, cols = np.nonzero(vacios)
        elegidos = rng.integers(0, k, size=filas.size)
        disponible[filas, cols, elegidos] = True

    severidad = rng.uniform(RANGO_FEATURES[0], RANGO_FEATURES[1], size=(n_usuarios, t_inter, k))
    impacto = rng.uniform(RANGO_FEATURES[0], RANGO_FEATURES[1], size=(n_usuarios, t_inter, k))
    confianza = rng.uniform(RANGO_FEATURES[0], RANGO_FEATURES[1], size=(n_usuarios, t_inter, k))
    u_respuesta = rng.random((n_usuarios, t_inter))
    return disponible, severidad, impacto, confianza, u_respuesta


def sub_ambiente(ambiente, filas):
    """Subconjunto de usuarios de un entorno (mantiene los mismos hallazgos)."""
    return tuple(componente[filas] for componente in ambiente)


# Verificación rápida del generador (una semilla).
_amb = generar_ambiente(np.random.default_rng(SEMILLA_BASE), 50, 5, K)
print("Formas:", [_a.shape for _a in _amb])
print("Fracción de tipos disponibles:", round(float(_amb[0].mean()), 3), "(esperado ≈ 0.5)")
print("Interacciones sin ningún hallazgo:", int((~_amb[0].any(axis=2)).sum()), "(debe ser 0)")


## 4. Las cinco políticas, en detalle

Además de las fórmulas de la sección 1:

* **`thompson_sin_prior`** usa `Beta(1 + utiles_usuario, 1 + no_utiles_usuario)`.
  Es el bandido «ingenuo»: arranca de una uniforme y solo aprende de los clics
  del propio usuario, que son escasos (en 30 interacciones un tipo se elige
  pocas veces).
* **`thompson_jerarquico`** usa las fórmulas de la sección 1. La diferencia
  clave es el **préstamo de información**: el prior poblacional arranca en
  `μ_diseno` y se actualiza con los clics de *todos* los usuarios simulados.
* **Actualización «en paralelo»**: para reproducir el orden temporal real, en
  la interacción `t` el prior poblacional usa únicamente las respuestas
  registradas hasta `t − 1`, y **todos los usuarios avanzan a la vez**
  (interacción por interacción), igual que ocurriría en producción.

> Nota de fidelidad: en la simulación la «población» son los 300 usuarios
> simulados del grupo bandido. En la app, la función SQL
> `estadisticas_bandido()` agrega a todos los usuarios que han respondido. La
> mecánica es idéntica; solo cambia el tamaño de la población.

* **`oraculo`** puntúa con la probabilidad *real* del usuario en lugar del peso
  aprendido: `p_real × severidad × impacto × confianza`. Marca el techo de lo
  alcanzable (solo como referencia, no es implementable).


In [ ]:
def simular_politica(politica, ambiente, P, rng_pol, M=M_PRIOR, guardar_usuario=None):
    """Ejecuta una política sobre un entorno y devuelve la traza completa.

    politica: 'control' | 'aleatoria' | 'thompson_sin_prior' |
              'thompson_jerarquico' | 'oraculo'
    P       : (n, k) probabilidad real de "útil" de cada usuario.
    rng_pol : generador propio de la política (no comparte estado con el entorno).
    M       : fuerza del prior del usuario (para el análisis de sensibilidad).
    guardar_usuario: índice de usuario cuya α/β se registra en cada interacción.
    """
    disponible, severidad, impacto, confianza, u_respuesta = ambiente
    n, t_max, k = disponible.shape
    filas = np.arange(n)

    util = np.zeros((n, t_max))          # 1 si el usuario marcó "útil"
    elegido = np.zeros((n, t_max), dtype=int)
    p_politica = np.zeros((n, t_max))    # p_real del tipo elegido
    p_oraculo = np.zeros((n, t_max))     # p_real del mejor tipo disponible

    utiles = np.zeros((n, k))            # conteos del usuario por tipo
    no_utiles = np.zeros((n, k))
    g_utiles = np.zeros(k)               # conteos de la población (todos los usuarios)
    g_no_utiles = np.zeros(k)

    hist_alfa, hist_beta = [], []

    for t in range(t_max):
        disp_t = disponible[:, t, :]

        if politica == "control":
            pesos = np.broadcast_to(PESO_CONTROL, (n, k))

        elif politica == "aleatoria":
            pesos = rng_pol.random((n, k))

        elif politica == "oraculo":
            pesos = P

        elif politica == "thompson_sin_prior":
            pesos = rng_pol.beta(1.0 + utiles, 1.0 + no_utiles)

        elif politica == "thompson_jerarquico":
            # --- Prior jerárquico con leave-one-out (bandido.ts) -------------
            pop_utiles = np.maximum(g_utiles[None, :] - utiles, 0.0)
            pop_no_utiles = np.maximum(g_no_utiles[None, :] - no_utiles, 0.0)
            mu_poblacion = (pop_utiles + C_DISENO * MU_VEC[None, :]) / \
                (pop_utiles + pop_no_utiles + C_DISENO)
            alfa = M * mu_poblacion + utiles
            beta = M * (1.0 - mu_poblacion) + no_utiles
            pesos = rng_pol.beta(alfa, beta)
            if guardar_usuario is not None:
                hist_alfa.append(alfa[guardar_usuario].copy())
                hist_beta.append(beta[guardar_usuario].copy())

        else:
            raise ValueError("Política desconocida: " + politica)

        # puntaje = peso_tipo × severidad × impacto × confianza (solo disponibles)
        puntaje = np.where(
            disp_t, pesos * severidad[:, t, :] * impacto[:, t, :] * confianza[:, t, :], -np.inf
        )
        elegido_t = np.argmax(puntaje, axis=1)

        p_elegida = P[filas, elegido_t]
        exito = (u_respuesta[:, t] < p_elegida).astype(float)

        util[:, t] = exito
        elegido[:, t] = elegido_t
        p_politica[:, t] = p_elegida
        # Referencia del oráculo: optimiza el MISMO puntaje que la app (severidad ×
        # impacto × confianza) pero con la probabilidad real en lugar del peso
        # aprendido. Por construcción, el oráculo tiene regret 0 frente a sí mismo.
        puntaje_oraculo = np.where(
            disp_t, P * severidad[:, t, :] * impacto[:, t, :] * confianza[:, t, :], -np.inf
        )
        p_oraculo[:, t] = P[filas, np.argmax(puntaje_oraculo, axis=1)]

        # Actualización: cada fila actualiza una única columna, así que += es seguro.
        utiles[filas, elegido_t] += exito
        no_utiles[filas, elegido_t] += 1.0 - exito

        # Los conteos globales incluyen a TODOS los usuarios hasta la interacción t
        # (avance en paralelo): el prior de t+1 usa lo ocurrido en 1..t.
        g_utiles = utiles.sum(axis=0)
        g_no_utiles = no_utiles.sum(axis=0)

    return {
        "util": util,
        "elegido": elegido,
        "p_politica": p_politica,
        "p_oraculo": p_oraculo,
        "alfa": np.array(hist_alfa),
        "beta": np.array(hist_beta),
    }


# Prueba mínima: una política aleatoria sólo debe elegir tipos disponibles.
_amb_prueba = generar_ambiente(np.random.default_rng(7), 20, 6, K)
_salida = simular_politica("aleatoria", _amb_prueba, P_REAL["A"][:20], np.random.default_rng(99))
_disponibles = _amb_prueba[0].mean(axis=2)
print("Elecciones válidas:", bool(np.all(
    _amb_prueba[0][np.arange(20)[:, None], np.arange(6)[None, :], _salida["elegido"]]
)))
print("Tasa de 'útil' de la política aleatoria (20 usuarios, 6 interacciones):",
      round(float(_salida["util"].mean()), 3))


## 5. Experimento principal

* 300 usuarios (100 por perfil) × 30 interacciones.
* Las 5 políticas se evalúan sobre **el mismo entorno** de cada semilla, y ese
  entorno se reutiliza en los **dos escenarios** (lo único que cambia es la
  probabilidad real de «útil» de cada usuario).
* Se repite con **50 semillas** distintas para estimar la variabilidad.
* Se guardan: la tasa de «útil» por interacción, la tasa acumulada por perfil y
  el arrepentimiento (regret) esperado frente al oráculo.

Las **figuras 1 a 4** de las secciones 7 a 10 muestran el **Escenario A
(«perfiles opuestos»)**; la comparación entre escenarios está en la Figura 5
(sección 13).


In [ ]:
POLITICAS = ["control", "aleatoria", "thompson_sin_prior", "thompson_jerarquico", "oraculo"]

# Resultados por semilla y por escenario:
#   tasa        : (n_semillas, T)        tasa de 'útil' en cada interacción
#   tasa_perfil : (n_semillas, 3, T)     idem, por perfil
#   regret      : (n_semillas, T)        regret esperado por interacción
res = {
    esc: {
        p: {
            "tasa": np.zeros((N_SEMILLAS, T_INTER)),
            "tasa_perfil": np.zeros((N_SEMILLAS, len(PERFILES), T_INTER)),
            "regret": np.zeros((N_SEMILLAS, T_INTER)),
        }
        for p in POLITICAS
    }
    for esc in ESCENARIOS
}

for s, semilla in enumerate(SEMILLAS):
    # El entorno (hallazgos y número aleatorio común) es el MISMO para las dos
    # poblaciones: así la comparación entre escenarios no se contamina con ruido.
    rng_entorno = np.random.default_rng(semilla)
    ambiente = generar_ambiente(rng_entorno, N_USUARIOS, T_INTER, K)

    for esc in ESCENARIOS:
        P = P_REAL[esc]
        for i, politica in enumerate(POLITICAS):
            rng_politica = np.random.default_rng(semilla * 1000 + i)
            salida = simular_politica(politica, ambiente, P, rng_politica)
            util = salida["util"]

            res[esc][politica]["tasa"][s] = util.mean(axis=0)
            for g in range(len(PERFILES)):
                res[esc][politica]["tasa_perfil"][s, g] = util[PERFIL_USUARIO == g].mean(axis=0)
            res[esc][politica]["regret"][s] = (salida["p_oraculo"] - salida["p_politica"]).mean(axis=0)

    if (s + 1) % 10 == 0:
        print("semillas completadas:", s + 1, "de", N_SEMILLAS)

print("\nSimulación principal terminada.")
for esc in ESCENARIOS:
    print("\n" + NOMBRE_ESCENARIO[esc])
    for politica in POLITICAS:
        acum = np.cumsum(res[esc][politica]["tasa"], axis=1) / np.arange(1, T_INTER + 1)
        print("  {:20s} tasa acumulada a 30: {:.3f}".format(politica, acum[:, T_INTER - 1].mean()))


## 6. Métricas

* **Tasa de útil por interacción `t`**: promedio de usuarios y semillas, con
  banda de IC 95 % (`media ± 1.96 · σ/√50`, con `σ` la desviación entre
  semillas).
* **Tasa acumulada hasta `t`**: promedio de las tasas de las interacciones
  1..t. Es la que se reporta a t = 5, 10 y 30.
* **Regret esperado**: en cada interacción,
  `p_real(mejor disponible) − p_real(elegido)`, medido en unidades de
  probabilidad de «útil» (es decir, en puntos de tasa). Se acumula suma a suma.
  El oráculo tiene regret 0 por definición.

La métrica se mide en la **probabilidad real** de utilidad (no en la respuesta
binaria observada) para que el regret sea una cantidad estable y no quede
dominada por el ruido de Bernoulli.


In [ ]:
def acumulada(serie):
    """Tasa acumulada hasta cada t. serie: (..., T) -> (..., T)."""
    return np.cumsum(serie, axis=-1) / np.arange(1, serie.shape[-1] + 1)


def media_ic(matriz, z=Z_IC):
    """Media y banda de IC 95 % a lo largo del eje 0 (las semillas)."""
    media = matriz.mean(axis=0)
    error = z * matriz.std(axis=0, ddof=1) / math.sqrt(matriz.shape[0])
    return media, media - error, media + error


def tasa_acumulada(escenario, politica, t, perfil=None):
    """Vector (por semilla) de la tasa acumulada hasta la interacción t."""
    if perfil is None:
        matriz = acumulada(res[escenario][politica]["tasa"])
    else:
        matriz = acumulada(res[escenario][politica]["tasa_perfil"])[:, PERFILES.index(perfil), :]
    return matriz[:, t - 1]


# Tabla resumen: tasa acumulada a 5, 10 y 30 por política, en cada escenario.
for esc in ESCENARIOS:
    filas_resumen = []
    for politica in POLITICAS:
        fila = {"política": politica}
        for t in (5, 10, 30):
            fila["t=%d" % t] = round(float(tasa_acumulada(esc, politica, t).mean()), 3)
        filas_resumen.append(fila)
    tabla_resumen = pd.DataFrame(filas_resumen).set_index("política")
    print(NOMBRE_ESCENARIO[esc])
    print(tabla_resumen)
    print()
    print("Regret acumulado al final del mes (en puntos de tasa de 'útil'):")
    for politica in POLITICAS:
        regret_final = np.cumsum(res[esc][politica]["regret"], axis=1)[:, T_INTER - 1].mean()
        print("  {:20s} {:.3f}".format(politica, regret_final))
    print()


## 7. Figura 1 — Tasa de utilidad por interacción

Cómo evoluciona cada política a lo largo del «mes». Lo esperable: la aleatoria
y el bandido sin prior parten bajos, el control arranca con el supuesto de
diseño y se queda plano, y el jerárquico converge hacia arriba.


In [ ]:
x = np.arange(1, T_INTER + 1)

fig, ax = plt.subplots(figsize=(9.5, 5.5))
for politica in ORDEN:
    media, inferior, superior = media_ic(res[ESC_A][politica]["tasa"])
    ax.plot(x, media, ESTILOS[politica], color=COLORES[politica],
            linewidth=2.2, label=NOMBRES[politica])
    ax.fill_between(x, inferior, superior, color=COLORES[politica], alpha=0.15, linewidth=0)

ax.set_xlabel("Interacción (día del mes)")
ax.set_ylabel("Tasa de recomendaciones marcadas «útil»")
ax.set_title("Tasa de utilidad por interacción y política\n(media de 50 semillas, 300 usuarios, banda IC 95 %)")
ax.set_xticks(np.arange(0, T_INTER + 1, 5))
ax.grid(alpha=0.3)
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(ruta("fig1_tasa_util_por_interaccion.png"), dpi=200)
plt.show()
print("Guardado:", ruta("fig1_tasa_util_por_interaccion.png"))


## 8. Figura 2 — Tasa acumulada a 30 interacciones, por perfil

Compara las políticas *dentro* de cada perfil. Aquí se ve el argumento central:
el peso fijo es el mismo para todos los perfiles, mientras que el bandido
jerárquico se adapta a lo que cada perfil valora.


In [ ]:
acum_perfil = {p: acumulada(res[ESC_A][p]["tasa_perfil"]) for p in POLITICAS}   # (S, 3, T)

ancho = 0.16
posiciones = np.arange(len(PERFILES))
fig, ax = plt.subplots(figsize=(10, 5.5))
for i, politica in enumerate(ORDEN):
    valores = acum_perfil[politica][:, :, T_INTER - 1].mean(axis=0)
    error = Z_IC * acum_perfil[politica][:, :, T_INTER - 1].std(axis=0, ddof=1) / math.sqrt(N_SEMILLAS)
    ax.bar(posiciones + (i - 2) * ancho, valores, ancho, yerr=error, capsize=3,
           color=COLORES[politica], edgecolor="white", linewidth=0.6,
           label=NOMBRES[politica])

ax.set_xticks(posiciones)
ax.set_xticklabels([p.capitalize() for p in PERFILES])
ax.set_xlabel("Perfil de usuario")
ax.set_ylabel("Tasa de «útil» acumulada a 30 interacciones")
ax.set_title("Tasa de utilidad a 30 interacciones por perfil y política\n(media de 50 semillas, banda IC 95 %)")
ax.grid(alpha=0.3, axis="y")
ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()
fig.savefig(ruta("fig2_tasa_util_por_perfil.png"), dpi=200)
plt.show()
print("Guardado:", ruta("fig2_tasa_util_por_perfil.png"))


## 9. Figura 3 — Arrepentimiento acumulado frente al oráculo

El regret acumulado mide cuánta utilidad esperada se «deja sobre la mesa» por
no conocer al usuario. Una curva más plana es mejor. La distancia vertical
entre el control y el jerárquico al final del mes es la ganancia de
aprendizaje.

El **oráculo** es la referencia: se define con el mismo puntaje que usa la app
(`p_real × severidad × impacto × confianza`) pero conociendo la probabilidad
real de cada usuario. Por construcción su regret es exactamente 0, y las demás
políticas se miden contra él.


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))
for politica in ORDEN:
    media, inferior, superior = media_ic(np.cumsum(res[ESC_A][politica]["regret"], axis=1))
    ax.plot(x, media, ESTILOS[politica], color=COLORES[politica],
            linewidth=2.2, label=NOMBRES[politica])
    ax.fill_between(x, inferior, superior, color=COLORES[politica], alpha=0.15, linewidth=0)

ax.set_xlabel("Interacción (día del mes)")
ax.set_ylabel("Arrepentimiento acumulado (puntos de tasa de «útil»)")
ax.set_title("Arrepentimiento acumulado frente al oráculo\n(media de 50 semillas, banda IC 95 %)")
ax.set_xticks(np.arange(0, T_INTER + 1, 5))
ax.grid(alpha=0.3)
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
fig.savefig(ruta("fig3_regret_acumulado.png"), dpi=200)
plt.show()
print("Guardado:", ruta("fig3_regret_acumulado.png"))


## 10. Figura 4 — Evolución de los pesos de un usuario «metas»

Se sigue a un usuario del perfil **metas** a lo largo de sus 30 interacciones y
se grafica `α/(α+β)` de cada tipo: es el **peso esperado** que el bandido le
asigna a cada tipo. Las líneas punteadas marcan los `μ_diseno` de partida.

Se espera que el peso del tipo que ese usuario valora (`meta`) suba por encima
de su valor de diseño y que los tipos que rechaza bajen. Es la evidencia visual
de que el prior jerárquico personaliza.


In [ ]:
# Un usuario del perfil "metas" (índice 1). Se registra su α y β en cada interacción.
indice_metas = int(np.where(PERFIL_USUARIO == 1)[0][0])

rng_entorno = np.random.default_rng(SEMILLAS[0])
ambiente_fig4 = generar_ambiente(rng_entorno, N_USUARIOS, T_INTER, K)
rng_politica = np.random.default_rng(SEMILLAS[0] * 1000 + 77)
salida_fig4 = simular_politica("thompson_jerarquico", ambiente_fig4, P_REAL[ESC_A], rng_politica,
                               guardar_usuario=indice_metas)

media_peso = salida_fig4["alfa"] / (salida_fig4["alfa"] + salida_fig4["beta"])   # (T, K)

fig, ax = plt.subplots(figsize=(9.5, 5.5))
for j, tipo in enumerate(TIPOS):
    ax.plot(x, media_peso[:, j], linewidth=2.2, label=ETIQUETA[tipo])
    ax.axhline(MU_DISENO[tipo], color=ax.lines[-1].get_color(), linestyle=":", alpha=0.6, linewidth=1.2)

ax.set_xlabel("Interacción (día del mes)")
ax.set_ylabel("Peso esperado del tipo  α / (α + β)")
ax.set_title("Evolución del peso esperado por tipo · usuario del perfil «metas»\n"
             "(punteado: μ_diseño de partida; política Thompson jerárquico)")
ax.set_xticks(np.arange(0, T_INTER + 1, 5))
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2, loc="upper right")
fig.tight_layout()
fig.savefig(ruta("fig4_evolucion_usuario.png"), dpi=200)
plt.show()

pesos_finales = pd.DataFrame(
    {"tipo": [ETIQUETA[t] for t in TIPOS],
     "p_real_usuario": P_REAL[ESC_A][indice_metas],
     "μ_diseño": MU_VEC,
     "peso_α/(α+β) al final": media_peso[T_INTER - 1]},
).set_index("tipo")
print("Pesos de ese usuario al final del mes:")
print(pesos_finales.round(3))
print("Guardado:", ruta("fig4_evolucion_usuario.png"))


## 11. Escenario del piloto real (honestidad estadística)

El A/B que se puede ejecutar de verdad es pequeño: **20 usuarios (10 control,
10 jerárquico), 10 interacciones** cada uno. Repetimos ese experimento **1.000
veces** para estimar:

* la **diferencia media** de tasa de útil entre grupos (bandido − control), y
* la **potencia**: proporción de repeticiones en las que una prueba de
  diferencia de proporciones (z para dos proporciones, bilateral) da `p < 0.05`.

Se usan 100 observaciones por grupo (10 usuarios × 10 interacciones).

**Qué significa para la tesis.** Con tan pocos usuarios la potencia será baja:
existe una probabilidad alta de no detectar como significativa una mejora real.
Por eso el A/B del piloto debe presentarse como **exploratorio** (estimación de
tamaño de efecto, usabilidad y aceptación), y las conclusiones cuantitativas
sobre la superioridad del bandido deben apoyarse en esta simulación, no en el
piloto.


In [ ]:
# --- Escenario del piloto real: 20 usuarios, 10 interacciones ---------------
USUARIOS_GRUPO = 10          # 10 control + 10 bandido
T_PILOTO = 10
REPETICIONES = 1000
N_PILOTO = 2 * USUARIOS_GRUPO


def normal_cdf(valor):
    return 0.5 * (1.0 + math.erf(valor / math.sqrt(2.0)))


def prueba_proporciones(exitos_a, n_a, exitos_b, n_b):
    """Prueba z bilateral de diferencia de proporciones (H0: p_a = p_b)."""
    p_a = exitos_a / n_a
    p_b = exitos_b / n_b
    p_comun = (exitos_a + exitos_b) / (n_a + n_b)
    error = math.sqrt(p_comun * (1 - p_comun) * (1 / n_a + 1 / n_b))
    if error <= 0:
        return 0.0, 1.0
    z = (p_b - p_a) / error
    return z, 2 * (1 - normal_cdf(abs(z)))


diferencias = np.zeros(REPETICIONES)
p_valores = np.zeros(REPETICIONES)
tasas_control = np.zeros(REPETICIONES)
tasas_bandido = np.zeros(REPETICIONES)

for r in range(REPETICIONES):
    rng = np.random.default_rng(SEMILLA_BASE + 500000 + r)

    # Los 20 usuarios del piloto se reparten entre los tres perfiles al azar.
    perfiles_piloto = rng.integers(0, len(PERFILES), size=N_PILOTO)
    P_piloto = np.array([[R_PROB_ESCENARIO[ESC_A][PERFILES[g]][t] for t in TIPOS] for g in perfiles_piloto])

    ambiente = generar_ambiente(rng, N_PILOTO, T_PILOTO, K)

    # Grupo control (usuarios 0..9) y grupo bandido (usuarios 10..19).
    util_control = simular_politica(
        "control", sub_ambiente(ambiente, np.arange(0, USUARIOS_GRUPO)),
        P_piloto[:USUARIOS_GRUPO], np.random.default_rng(SEMILLA_BASE + 600000 + r))["util"]
    util_bandido = simular_politica(
        "thompson_jerarquico", sub_ambiente(ambiente, np.arange(USUARIOS_GRUPO, N_PILOTO)),
        P_piloto[USUARIOS_GRUPO:], np.random.default_rng(SEMILLA_BASE + 700000 + r))["util"]

    exitos_control = float(util_control.sum())
    exitos_bandido = float(util_bandido.sum())
    n_observaciones = util_control.size

    tasas_control[r] = exitos_control / n_observaciones
    tasas_bandido[r] = exitos_bandido / n_observaciones
    diferencias[r] = tasas_bandido[r] - tasas_control[r]
    p_valores[r] = prueba_proporciones(exitos_control, n_observaciones,
                                      exitos_bandido, n_observaciones)[1]

potencia = float((p_valores < 0.05).mean())

# IC 95 % de la diferencia media entre repeticiones.
error_dif = Z_IC * diferencias.std(ddof=1) / math.sqrt(REPETICIONES)
piloto_resumen = pd.DataFrame([{
    "escenario": "piloto_20_usuarios_10_interacciones",
    "usuarios_por_grupo": USUARIOS_GRUPO,
    "interacciones_por_usuario": T_PILOTO,
    "repeticiones": REPETICIONES,
    "observaciones_por_grupo": USUARIOS_GRUPO * T_PILOTO,
    "tasa_control_media": round(float(tasas_control.mean()), 4),
    "tasa_bandido_media": round(float(tasas_bandido.mean()), 4),
    "diferencia_media": round(float(diferencias.mean()), 4),
    "ic95_diferencia_inferior": round(float(diferencias.mean() - error_dif), 4),
    "ic95_diferencia_superior": round(float(diferencias.mean() + error_dif), 4),
    "potencia_p_menor_0.05": round(potencia, 4),
    "prop_repeticiones_bandido_superior": round(float((diferencias > 0).mean()), 4),
}])

print("Resultado del piloto simulado (", REPETICIONES, "repeticiones ):")
print(piloto_resumen.T.to_string())
print("\nLectura: la potencia es la probabilidad de detectar la mejora con 10 usuarios por grupo.")


## 12. Sensibilidad al parámetro `M` y «% de la brecha cerrada»

`M = M_PRIOR` es la fuerza del prior del usuario: cuántas «respuestas virtuales»
aporta la población antes de ver ningún clic propio.

* `M` grande → más confianza en la población: aprendizaje lento pero estable.
* `M` pequeño → más confianza en el usuario: aprende rápido pero con ruido.

Se repite `thompson_jerarquico` con **M = 1, 2, 4 y 10** sobre las mismas 50
semillas y en **los dos escenarios**, junto con `control`, `aleatoria`,
`thompson_sin_prior` y `oraculo`. `M = 4` es el valor de `M_PRIOR` en
`config.ts`, así que los resultados con `M = 4` coinciden con el experimento
principal de la sección 5 (mismo flujo aleatorio).

### «% de la brecha cerrada»

El `control` (peso fijo) es el punto de partida y el `oraculo` es el techo. La
distancia entre ambos es toda la utilidad que se puede ganar conociendo al
usuario. La columna **«% de la brecha cerrada»** mide qué fracción de esa
distancia captura cada política:

```
% brecha cerrada = (política − control) / (oráculo − control) × 100
```

* `control` → 0 % por definición; `oráculo` → 100 % por definición.
* **Negativo** → la política queda *peor* que el peso fijo.
* Cerca de 100 % → la política ya casi extrae todo lo que se puede extraer.
* Comparar el % entre escenarios es más informativo que comparar tasas
  absolutas, porque normaliza por la dificultad de cada población.

Se exporta a `ia/fase5/sensibilidad_M.csv` (escenario × política × interacción,
con media e IC 95 %).


In [ ]:
# --- Sensibilidad a M en los dos escenarios ---------------------------------
# Se reutiliza el resultado de M = M_PRIOR (4) del experimento principal: el
# bucle principal lanzó thompson_jerarquico con rng_politica = semilla*1000+3.
IDX_JERARQUICO = POLITICAS.index("thompson_jerarquico")
sens_tasas = {esc: {M_PRIOR: res[esc]["thompson_jerarquico"]["tasa"]} for esc in ESCENARIOS}

for esc in ESCENARIOS:
    for valor_m in [m for m in M_VALORES if m != M_PRIOR]:
        tasas_m = np.zeros((N_SEMILLAS, T_INTER))
        for s, semilla in enumerate(SEMILLAS):
            rng_entorno = np.random.default_rng(semilla)
            ambiente = generar_ambiente(rng_entorno, N_USUARIOS, T_INTER, K)
            rng_politica = np.random.default_rng(semilla * 1000 + IDX_JERARQUICO)
            salida = simular_politica("thompson_jerarquico", ambiente, P_REAL[esc],
                                      rng_politica, M=valor_m)
            tasas_m[s] = salida["util"].mean(axis=0)
        sens_tasas[esc][valor_m] = tasas_m
    print("Escenario", esc, "· M evaluados:", M_VALORES)


# --- Tabla resumen: escenario × política (incluyendo cada M) × interacción ----
FILAS_POLITICAS = [
    ("control", "control", None),
    ("aleatoria", "aleatoria", None),
    ("thompson_sin_prior", "thompson_sin_prior", None),
] + [("thompson_jerarquico_M%d" % m, "thompson_jerarquico", m) for m in M_VALORES] + [
    ("oraculo", "oraculo", None),
]


def serie_tasa(escenario, politica, m):
    """Matriz (n_semillas, T) de tasa de «útil» por interacción."""
    if politica == "thompson_jerarquico" and m is not None:
        return sens_tasas[escenario][m]
    return res[escenario][politica]["tasa"]


def media_ic_columna(columna, z=Z_IC):
    """Media e IC 95 % de una columna (una observación por semilla)."""
    media = float(columna.mean())
    error = z * float(columna.std(ddof=1)) / math.sqrt(columna.shape[0])
    return media, media - error, media + error


filas = []
for esc in ESCENARIOS:
    control_acum = acumulada(serie_tasa(esc, "control", None))
    oraculo_acum = acumulada(serie_tasa(esc, "oraculo", None))
    for etiqueta, politica, m in FILAS_POLITICAS:
        series = acumulada(serie_tasa(esc, politica, m))
        for t in (5, 10, 30):
            media, inf, sup = media_ic_columna(series[:, t - 1])
            control_t = float(control_acum[:, t - 1].mean())
            oraculo_t = float(oraculo_acum[:, t - 1].mean())
            filas.append({
                "escenario": esc,
                "politica": etiqueta,
                "M": m,
                "interaccion": t,
                "tasa_media": round(media, 4),
                "ic_inferior": round(inf, 4),
                "ic_superior": round(sup, 4),
                "brecha_cerrada_pct": round(100.0 * (media - control_t) / (oraculo_t - control_t), 1),
            })

sensibilidad_M = pd.DataFrame(filas)
sensibilidad_M["M"] = sensibilidad_M["M"].astype("Int64")
sensibilidad_M.to_csv(ruta("sensibilidad_M.csv"), index=False, encoding="utf-8")

print()
print("Sensibilidad al prior M por escenario (tasa acumulada media, 50 semillas):")
for esc in ESCENARIOS:
    sub = sensibilidad_M[sensibilidad_M["escenario"] == esc]
    pivote_tasa = sub.pivot(index="politica", columns="interaccion", values="tasa_media")
    pivote_brecha = sub.pivot(index="politica", columns="interaccion", values="brecha_cerrada_pct")
    tabla = pd.DataFrame({
        "tasa a 5": pivote_tasa[5], "tasa a 10": pivote_tasa[10], "tasa a 30": pivote_tasa[30],
        "brecha% a 5": pivote_brecha[5], "brecha% a 10": pivote_brecha[10], "brecha% a 30": pivote_brecha[30],
    })
    print()
    print(NOMBRE_ESCENARIO[esc])
    print(tabla.round(3))
print()
print("Guardado:", ruta("sensibilidad_M.csv"), "->", sensibilidad_M.shape[0], "filas")


## 13. Figura 5 — Sensibilidad a `M`, por escenario

Dos paneles: **(A)** «perfiles opuestos» y **(B)** «población con tendencia
común». En cada panel se compara la **tasa acumulada** interacción a interacción
para el peso fijo (`control`), el bandido ingenuo (`thompson_sin_prior`) y el
jerárquico con `M = 1, 2, 4`.

Sirve para ver dos cosas de un vistazo: cuánto cambia el resultado al mover `M`,
y en qué escenario el prior poblacional aporta utilidad de verdad (en A aporta
poco porque la población es heterogénea; en B aporta más porque hay una
tendencia común).


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5.4), sharey=True)

COLORES_M = {1: "#93c5fd", 2: "#3b82f6", 4: "#1d4ed8"}

for eje, esc in zip(ejes, ESCENARIOS):
    eje.plot(x, acumulada(res[esc]["control"]["tasa"]).mean(axis=0),
             ESTILOS["control"], color=COLORES["control"], linewidth=2.2,
             label=NOMBRES["control"])
    eje.plot(x, acumulada(res[esc]["thompson_sin_prior"]["tasa"]).mean(axis=0),
             ESTILOS["thompson_sin_prior"], color=COLORES["thompson_sin_prior"],
             linewidth=2.2, label=NOMBRES["thompson_sin_prior"])
    for m in [1, 2, 4]:
        eje.plot(x, acumulada(sens_tasas[esc][m]).mean(axis=0),
                 color=COLORES_M[m], linewidth=2.2,
                 label="Thompson jerárquico (M = %d)" % m)

    eje.set_title(NOMBRE_ESCENARIO[esc])
    eje.set_xlabel("Interacción (día del mes)")
    eje.set_xticks(np.arange(0, T_INTER + 1, 5))
    eje.grid(alpha=0.3)
    eje.legend(fontsize=8, loc="lower right")

ejes[0].set_ylabel("Tasa de «útil» acumulada")
fig.suptitle("Sensibilidad al prior M · tasa acumulada por escenario\n"
             "(media de 50 semillas, 300 usuarios)", y=1.01)
fig.tight_layout()
fig.savefig(ruta("fig5_sensibilidad_M.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Guardado:", ruta("fig5_sensibilidad_M.png"))


## 14. Lectura del «% de la brecha cerrada»

Los números de esta sección son los de la ejecución documentada en
`ia/fase5/sensibilidad_M.csv`. Como las semillas están fijas, son
**reproducibles**: volver a ejecutar el cuaderno produce estos mismos valores.
`M = 4` es el valor que usa hoy `M_PRIOR` en `config.ts`.

### Escenario A · «perfiles opuestos»

| Política | tasa a 5 | brecha a 5 | tasa a 10 | brecha a 10 | tasa a 30 | brecha a 30 |
|---|---|---|---|---|---|---|
| control (peso fijo) | 0.511 | 0.0 % | 0.511 | 0.0 % | 0.511 | 0.0 % |
| aleatoria | 0.504 | −6.9 % | 0.504 | −7.1 % | 0.503 | −7.3 % |
| thompson sin prior | 0.516 | +5.4 % | 0.527 | +15.9 % | 0.551 | +40.4 % |
| jerárquico M = 1 | 0.520 | +9.5 % | 0.534 | +22.8 % | 0.559 | +47.9 % |
| jerárquico M = 2 | 0.519 | +7.9 % | 0.530 | +18.3 % | 0.552 | +40.9 % |
| jerárquico M = 4 | 0.514 | +3.4 % | 0.522 | +11.0 % | 0.542 | +31.3 % |
| jerárquico M = 10 | 0.512 | +1.4 % | 0.516 | +5.2 % | 0.530 | +18.8 % |
| oráculo (techo) | 0.612 | +100.0 % | 0.612 | +100.0 % | 0.612 | +100.0 % |

### Escenario B · «población con tendencia común»

| Política | tasa a 5 | brecha a 5 | tasa a 10 | brecha a 10 | tasa a 30 | brecha a 30 |
|---|---|---|---|---|---|---|
| control (peso fijo) | 0.581 | 0.0 % | 0.581 | 0.0 % | 0.582 | 0.0 % |
| aleatoria | 0.552 | −57.0 % | 0.553 | −56.0 % | 0.553 | −57.5 % |
| thompson sin prior | 0.561 | −40.3 % | 0.569 | −23.4 % | 0.589 | +12.7 % |
| jerárquico M = 1 | 0.588 | +12.0 % | 0.592 | +21.0 % | 0.604 | +43.4 % |
| jerárquico M = 2 | 0.588 | +12.0 % | 0.592 | +21.9 % | 0.603 | +40.6 % |
| jerárquico M = 4 | 0.589 | +15.1 % | 0.593 | +23.4 % | 0.601 | +37.6 % |
| jerárquico M = 10 | 0.591 | +19.1 % | 0.594 | +24.3 % | 0.599 | +33.4 % |
| oráculo (techo) | 0.632 | +100.0 % | 0.632 | +100.0 % | 0.633 | +100.0 % |

### Qué se lee en estos números

**El «% de brecha cerrada» normaliza por la dificultad del escenario.** En el
Escenario A la distancia oráculo − control es grande (0.612 − 0.511 = 0.101
puntos) porque el peso fijo cae lejos de lo que valora cada perfil; en el B es
mucho menor (0.632 − 0.582 = 0.050 puntos) porque `μ_diseño` ya encaja
razonablemente con la tendencia común. Por eso el mismo `M` puede tener un
porcentaje muy distinto en cada escenario sin que ello signifique otra cosa que
un denominador distinto.

**En A hay mucho que ganar y el prior poblacional estorba.** El jerárquico con
`M = 1` cierra el **47.9 %** de la brecha a 30 interacciones, por encima del
bandido sin prior (**40.4 %**). Al subir `M` el resultado empeora de forma
monótona: `M = 4` cierra 31.3 % y `M = 10`, apenas 18.8 %. La causa es el propio
prior: `μ_diseño` es una media de tipos que en A no representa a nadie, y cuanto
más se confía en ella, más tarda el usuario en «corregirla» con sus clics.

**En B el prior ayuda desde el principio.** A las 5 interacciones el jerárquico
ya cierra entre +12.0 % y +19.1 %, mientras el sin prior está **en negativo**
(−40.3 %): un usuario nuevo sin clics aún no tiene nada con qué decidir, pero la
población sí. A las 30 interacciones el orden se invierte: `M = 1` cierra
+43.4 % y `M = 10` cierra +33.4 %. Es decir, **un `M` alto compra rapidez al
principio y la paga con peor ajuste al final**; un `M` bajo arranca un poco peor
pero acaba mejor.

**La aleatoria mide cuánto importa el orden.** En B cierra −57.5 %, es decir,
elegir al azar es mucho peor que el peso fijo, porque el peso fijo ya está bien
alineado con la población. En A, en cambio, cierra solo −7.3 %: cuando el peso
fijo es malo para todos, elegir al azar apenas empeora las cosas. Esto confirma
que la utilidad del control no es un valor fijo, sino que depende del escenario.

**Conclusión de la lectura.** El `%` de brecha cerrada es la métrica comparable
entre escenarios: en A el mejor es `M = 1` (+47.9 %) y el peor `M = 10`
(+18.8 %); en B el mejor a 30 también es `M = 1` (+43.4 %), aunque a 5
interacciones gane `M = 10` (+19.1 %). El denominador común es que `M = 1` nunca
queda por debajo de la política sin prior en ninguno de los dos escenarios,
mientras que `M = 4` (el valor actual de `config.ts`) sí lo hace en el
Escenario A. La recomendación cuantitativa se cierra en la sección de
conclusiones.


## 15. Exportación de resultados

* `resultados_fase5.csv`: por política, perfil e interacción (5, 10 y 30), tasa
  acumulada media e IC 95 %.
* `potencia_piloto.csv`: resumen del escenario del piloto (diferencia media y
  potencia).
* `sensibilidad_M.csv`: escenario × política (incluyendo cada `M`) × interacción
  (5, 10 y 30), con media, IC 95 % y «% de la brecha cerrada» (sección 12).


In [ ]:
filas_csv = []
for politica in POLITICAS:
    acum_global = acumulada(res[ESC_A][politica]["tasa"])
    acum_perfiles = acumulada(res[ESC_A][politica]["tasa_perfil"])
    for t in (5, 10, 30):
        escenarios = [("todos", acum_global)] + [
            (PERFILES[g], acum_perfiles[:, g, :]) for g in range(len(PERFILES))
        ]
        for nombre_perfil, matriz in escenarios:
            columna = matriz[:, t - 1]
            media = float(columna.mean())
            error = Z_IC * float(columna.std(ddof=1)) / math.sqrt(N_SEMILLAS)
            filas_csv.append({
                "politica": politica,
                "perfil": nombre_perfil,
                "interaccion": t,
                "tasa_media": round(media, 4),
                "ic_inferior": round(media - error, 4),
                "ic_superior": round(media + error, 4),
            })

resultados = pd.DataFrame(filas_csv)
resultados.to_csv(ruta("resultados_fase5.csv"), index=False, encoding="utf-8")
piloto_resumen.to_csv(ruta("potencia_piloto.csv"), index=False, encoding="utf-8")

print("Archivos escritos:")
print("  ", ruta("resultados_fase5.csv"), "->", resultados.shape[0], "filas")
print("  ", ruta("potencia_piloto.csv"), "->", piloto_resumen.shape[0], "fila(s)")
print()
print(resultados.head(8).to_string(index=False))


## 16. Conclusiones para la tesis

La celda siguiente **calcula e imprime** las conclusiones con los números
obtenidos en esta ejecución (no están escritos a mano). Sirve de borrador para
el capítulo de resultados.


In [ ]:
# ============================================================================
# Conclusiones calculadas automáticamente (los números salen de esta ejecución)
# ============================================================================
def curva_acum(esc, politica, m=None):
    """Curva (T,) de la tasa acumulada media (promedio de las 50 semillas)."""
    return acumulada(serie_tasa(esc, politica, m)).mean(axis=0)


def valor(esc, politica, m, t):
    """Tasa acumulada media del escenario, política y M dados, hasta t."""
    return float(curva_acum(esc, politica, m)[t - 1])


def brecha_pct(esc, politica, m, t):
    """% de la brecha cerrada frente al oráculo, usando el control como base."""
    control_t = valor(esc, "control", None, t)
    oraculo_t = valor(esc, "oraculo", None, t)
    return 100.0 * (valor(esc, politica, m, t) - control_t) / (oraculo_t - control_t)


print("=" * 84)
print("CONCLUSIONES PARA LA TESIS")
print("=" * 84)
print()
print("Diseño: {} usuarios ({} por perfil) x {} interacciones x {} semillas, sobre dos".format(
    N_USUARIOS, USUARIOS_POR_PERFIL, T_INTER, N_SEMILLAS))
print("escenarios de población: A «perfiles opuestos» y B «población con tendencia común».")
print("Las probabilidades reales de «útil» son supuestos de diseño (sección 2).")
print()

for esc in ESCENARIOS:
    print("=" * 84)
    print(NOMBRE_ESCENARIO[esc])
    print("=" * 84)
    print()
    print("1) Tasa de «útil» acumulada y % de brecha cerrada (0 % = control, 100 % = oráculo):")
    print("   {:26s} {:>18s} {:>18s} {:>18s}".format("política", "a 5", "a 10", "a 30"))
    for etiqueta, politica, m in FILAS_POLITICAS:
        celdas = " ".join(
            "{:.3f} ({:+5.1f} %)".format(valor(esc, politica, m, t), brecha_pct(esc, politica, m, t))
            for t in (5, 10, 30)
        )
        print("   {:26s} {}".format(etiqueta, celdas))
    print()
    mejor_5 = max(M_VALORES, key=lambda m: valor(esc, "thompson_jerarquico", m, 5))
    mejor_10 = max(M_VALORES, key=lambda m: valor(esc, "thompson_jerarquico", m, 10))
    mejor_30 = max(M_VALORES, key=lambda m: valor(esc, "thompson_jerarquico", m, 30))
    print("2) Mejor M de thompson_jerarquico en ESTE escenario:")
    print("     a 5 interacciones:  M = {:2d}  ({:.3f})".format(
        mejor_5, valor(esc, "thompson_jerarquico", mejor_5, 5)))
    print("     a 10 interacciones: M = {:2d}  ({:.3f})".format(
        mejor_10, valor(esc, "thompson_jerarquico", mejor_10, 10)))
    print("     a 30 interacciones: M = {:2d}  ({:.3f})".format(
        mejor_30, valor(esc, "thompson_jerarquico", mejor_30, 30)))
    print()
    sp_30 = valor(esc, "thompson_sin_prior", None, 30)
    print("3) Bandido ingenuo (thompson_sin_prior) a 30: {:.3f}".format(sp_30))
    for m in M_VALORES:
        print("     jerárquico M = {:2d} -> {:.3f} ({:+.3f} frente al ingenuo) | brecha cerrada {:+5.1f} %".format(
            m, valor(esc, "thompson_jerarquico", m, 30),
            valor(esc, "thompson_jerarquico", m, 30) - sp_30,
            brecha_pct(esc, "thompson_jerarquico", m, 30)))
    print()

print("=" * 84)
print("¿QUÉ M SE RECOMIENDA?")
print("=" * 84)
print()
print("Criterio: un M es «seguro» si su tasa acumulada media NUNCA queda por debajo")
print("de thompson_sin_prior, ni en un escenario ni en una interacción (t = 1..30).")
print()
seguros = []
for m in M_VALORES:
    deficits = {
        esc: float(np.min(curva_acum(esc, "thompson_jerarquico", m)
                          - curva_acum(esc, "thompson_sin_prior", None)))
        for esc in ESCENARIOS
    }
    seguro = all(v >= 0 for v in deficits.values())
    if seguro:
        seguros.append(m)
    print("  M = {:2d}  déficit mínimo frente a sin_prior: {}".format(
        m, " | ".join("{} {:+.4f}".format(esc, deficits[esc]) for esc in ESCENARIOS)),
        "->", "nunca por debajo" if seguro else "queda por debajo")
print()

if seguros:
    promedio = {m: float(np.mean([valor(esc, "thompson_jerarquico", m, 30) for esc in ESCENARIOS]))
                for m in seguros}
    recomendado = max(promedio, key=promedio.get)
    print("M seguros (nunca por debajo del ingenuo en ningún escenario):", seguros)
    print("Tasa acumulada media a 30, promediando los dos escenarios:")
    for m in seguros:
        print("     M = {:2d} -> {:.3f}".format(m, promedio[m]))
    print()
    print("=> SE RECOMIENDA M = {}: es el mejor de los M seguros y nunca queda por".format(recomendado))
    print("   debajo del bandido sin prior en ninguno de los dos escenarios.")
else:
    minimos = {
        m: min(float(np.min(curva_acum(esc, "thompson_jerarquico", m)
                            - curva_acum(esc, "thompson_sin_prior", None))) for esc in ESCENARIOS)
        for m in M_VALORES
    }
    recomendado = max(minimos, key=minimos.get)
    print("Ningún M cumple el criterio estricto de no quedar nunca por debajo.")
    print("=> SE RECOMIENDA M = {}: es el que menos cae por debajo del ingenuo".format(recomendado))
    print("   (déficit mínimo observado {:+.4f}).".format(minimos[recomendado]))
print()
print("4) Brecha cerrada a 30 interacciones (100 % = oráculo, 0 % = control):")
for esc in ESCENARIOS:
    lecturas = ", ".join(
        "{} {:+5.1f} %".format(etiqueta, brecha_pct(esc, politica, m, 30))
        for etiqueta, politica, m in FILAS_POLITICAS
    )
    print("     Escenario {}: {}".format(esc, lecturas))
print()
print("5) Piloto real simulado ({} usuarios, {} interacciones, {} repeticiones):".format(
    USUARIOS_GRUPO * 2, T_PILOTO, REPETICIONES))
print("     corresponde al Escenario A (sección 11).")
print("     diferencia media bandido - control: {:+.4f} puntos de tasa".format(
    float(piloto_resumen["diferencia_media"].iloc[0])))
print("     IC 95 % de la diferencia: [{:+.4f}, {:+.4f}]".format(
    float(piloto_resumen["ic95_diferencia_inferior"].iloc[0]),
    float(piloto_resumen["ic95_diferencia_superior"].iloc[0])))
print("     potencia (p < 0.05): {:.1f} %".format(
    100.0 * float(piloto_resumen["potencia_p_menor_0.05"].iloc[0])))
print("     -> con el tamaño del piloto NO se detectaría como significativa una mejora")
print("        real; el A/B debe presentarse como exploratorio.")
print()
print("6) Lectura general:")
print("     - La evidencia cuantitativa proviene de esta simulación ({} usuarios x {} interacciones".format(
    N_USUARIOS, T_INTER))
print("       x {} semillas), no del piloto.".format(N_SEMILLAS))
print("     - Los valores absolutos dependen de los supuestos de diseño de la sección 2 y de")
print("       cuánto se parezca la población real (A vs B); deben recalibrarse con datos reales.")
print("=" * 84)
